# NumPy Pratik: Öğrenci Performans Veri Seti Analizi

Bu notebook, **NumPy Temelleri** dersinde öğrenilen kavramların gerçek bir veri seti üzerinde uygulamasıdır.

**Veri seti:** `Student_Performance_Dataset.csv` — 5.000 öğrencinin demografik bilgileri, çalışma alışkanlıkları ve sınav notları.

Derste öğrendiğimiz şu konuları burada pratik yapacağız:
1. Veriyi NumPy array'ine dönüştürme
2. Array özellikleri (shape, dtype, ndim)
3. İndeksleme ve dilimleme
4. Boolean (mantıksal) indeksleme ile filtreleme
5. Toplulaştırma (aggregate) fonksiyonlarıyla istatistikler
6. Broadcasting ile hesaplamalar
7. Reshape ve rastgele örneklem

## 1. Veriyi Yükleme

Önce CSV dosyasını okuyup sayısal sütunları NumPy array'ine dönüştürüyoruz.

In [1]:
# numpy kütüphanesini "np" takma adıyla içe aktarıyoruz (topluluk standardı)
import numpy as np
import csv

print("NumPy versiyonu:", np.__version__)

NumPy versiyonu: 2.4.4


In [2]:
# CSV dosyasını satır satır okuyoruz
with open("Student_Performance_Dataset.csv", "r") as f:
    okuyucu = csv.reader(f)
    basliklar = next(okuyucu)          # ilk satır sütun isimleri
    satirlar = list(okuyucu)           # geri kalan tüm satırlar

print("Sütunlar:", basliklar)
print("Toplam öğrenci sayısı:", len(satirlar))

Sütunlar: ['Student_ID', 'Age', 'Gender', 'Class', 'Study_Hours_Per_Day', 'Attendance_Percentage', 'Parental_Education', 'Internet_Access', 'Extracurricular_Activities', 'Math_Score', 'Science_Score', 'English_Score', 'Previous_Year_Score', 'Final_Percentage', 'Performance_Level', 'Pass_Fail']
Toplam öğrenci sayısı: 5000


In [3]:
# Sayısal sütunları seçip NumPy array'lerine dönüştürüyoruz
# Sütun indeksleri: 1=Age, 4=Study_Hours, 5=Attendance, 9=Math, 10=Science, 11=English, 13=Final_Percentage

yas          = np.array([int(s[1])     for s in satirlar])   # yaş -> tam sayı (int)
calisma_saat = np.array([float(s[4])   for s in satirlar])   # günlük çalışma saati -> ondalıklı (float)
devam        = np.array([float(s[5])   for s in satirlar])   # devam yüzdesi
matematik    = np.array([int(s[9])     for s in satirlar])   # matematik notu
fen          = np.array([int(s[10])    for s in satirlar])   # fen notu
ingilizce    = np.array([int(s[11])    for s in satirlar])   # İngilizce notu
final_yuzde  = np.array([float(s[13])  for s in satirlar])   # yıl sonu başarı yüzdesi

# Kategorik sütunlar (metin olarak kalacak)
cinsiyet   = np.array([s[2]  for s in satirlar])   # Male / Female
gecti_kaldi = np.array([s[15] for s in satirlar])  # Pass / Fail

print("Matematik notları (ilk 10):", matematik[:10])

Matematik notları (ilk 10): [40 80 83 68 41 88 75 93 55 72]


## 2. Array Özellikleri

Derste gördüğümüz `shape`, `dtype`, `ndim`, `size` özniteliklerini kendi verimizde inceliyoruz.

In [4]:
print("shape :", matematik.shape)   # kaç elemanlı olduğunu gösterir
print("ndim  :", matematik.ndim)    # boyut sayısı (1D)
print("size  :", matematik.size)    # toplam eleman sayısı
print("dtype :", matematik.dtype)   # veri tipi (int64)
print()
print("Çalışma saati dtype:", calisma_saat.dtype)  # float64 -> ondalıklı değerler

shape : (5000,)
ndim  : 1
size  : 5000
dtype : int64

Çalışma saati dtype: float64


In [5]:
# Üç dersin notlarını tek bir 2D matris haline getirelim (derste gördüğümüz 2D array mantığı)
# np.column_stack: 1D array'leri sütun sütun yan yana dizerek matris oluşturur
notlar = np.column_stack([matematik, fen, ingilizce])

print("Notlar matrisi shape:", notlar.shape)   # (5000, 3) -> 5000 satır (öğrenci), 3 sütun (ders)
print("İlk 5 öğrencinin notları:")
print(notlar[:5])

Notlar matrisi shape: (5000, 3)
İlk 5 öğrencinin notları:
[[40 39 72]
 [80 44 35]
 [83 73 59]
 [68 48 77]
 [41 46 36]]


## 3. İndeksleme ve Dilimleme (Slicing)

Derste `a[0]`, `a[1:4]`, `m[satır, sütun]` gibi kullanımları görmüştük. Şimdi gerçek veride uyguluyoruz.

In [6]:
print("İlk öğrencinin matematik notu :", matematik[0])
print("Son öğrencinin matematik notu :", matematik[-1])    # negatif indeks sondan sayar
print("5-10 arası öğrencilerin notları:", matematik[5:10]) # dilimleme (slicing)
print()
# 2D matriste indeksleme: [satır, sütun]
print("3. öğrencinin fen notu        :", notlar[2, 1])     # 2. indeks satır=3. öğrenci, 1. indeks sütun=fen
print("İlk 3 öğrencinin tüm notları  :\n", notlar[:3, :])
print("Tüm öğrencilerin İngilizce sütunu (ilk 10):", notlar[:10, 2])

İlk öğrencinin matematik notu : 40
Son öğrencinin matematik notu : 69
5-10 arası öğrencilerin notları: [88 75 93 55 72]

3. öğrencinin fen notu        : 73
İlk 3 öğrencinin tüm notları  :
 [[40 39 72]
 [80 44 35]
 [83 73 59]]
Tüm öğrencilerin İngilizce sütunu (ilk 10): [72 35 59 77 36 46 64 48 83 73]


## 4. Boolean İndeksleme ile Filtreleme

Derste `a[a > 3]` şeklinde koşula uyan elemanları seçmeyi öğrenmiştik. Bu, veri analizinde en çok kullanılan tekniklerden biridir.

In [7]:
# Matematikten 80 ve üzeri alan öğrenciler
basarili_mat = matematik[matematik >= 80]
print("Matematikten 80+ alan öğrenci sayısı:", basarili_mat.size)
print("Bu öğrencilerin oranı: %", round(basarili_mat.size / matematik.size * 100, 1))

Matematikten 80+ alan öğrenci sayısı: 1588
Bu öğrencilerin oranı: % 31.8


In [8]:
# Birden fazla koşulu & (ve) ile birleştirebiliriz (her koşul parantez içinde olmalı!)
# Hem matematikten hem fenden 80+ alan öğrenciler
cift_basarili = (matematik >= 80) & (fen >= 80)
print("Hem matematik hem fen 80+ olan öğrenci sayısı:", cift_basarili.sum())  # True'lar 1 sayılır

# | (veya) operatörü: en az bir dersten 90+ alanlar
en_az_bir_90 = (matematik >= 90) | (fen >= 90) | (ingilizce >= 90)
print("En az bir dersten 90+ alan öğrenci sayısı  :", en_az_bir_90.sum())

Hem matematik hem fen 80+ olan öğrenci sayısı: 469
En az bir dersten 90+ alan öğrenci sayısı  : 2121


In [9]:
# Boolean maskeyi başka bir array'e uygulama:
# Günde 4 saatten fazla çalışan öğrencilerin final yüzdesi ortalaması
cok_calisanlar = final_yuzde[calisma_saat > 4]
az_calisanlar  = final_yuzde[calisma_saat < 2]

print("4+ saat çalışanların ortalama başarısı: %", round(cok_calisanlar.mean(), 2))
print("2 saatten az çalışanların ortalaması  : %", round(az_calisanlar.mean(), 2))

4+ saat çalışanların ortalama başarısı: % 67.19
2 saatten az çalışanların ortalaması  : % 67.58


In [10]:
# Kategorik sütunla filtreleme: kalan (Fail) öğrencilerin devam yüzdesi
kalanlarin_devami = devam[gecti_kaldi == "Fail"]
gecenlerin_devami = devam[gecti_kaldi == "Pass"]

print("Kalan öğrenci sayısı:", kalanlarin_devami.size)
print("Kalanların ortalama devam yüzdesi : %", round(kalanlarin_devami.mean(), 1))
print("Geçenlerin ortalama devam yüzdesi : %", round(gecenlerin_devami.mean(), 1))

Kalan öğrenci sayısı: 265
Kalanların ortalama devam yüzdesi : % 75.6
Geçenlerin ortalama devam yüzdesi : % 74.9


## 5. Toplulaştırma (Aggregate) Fonksiyonları

Derste `sum`, `mean`, `min`, `max`, `std`, `argmax` fonksiyonlarını görmüştük. Şimdi veri setimizin genel istatistiklerini çıkarıyoruz.

In [11]:
print("=== Matematik Notu İstatistikleri ===")
print("Ortalama       :", round(matematik.mean(), 2))
print("En düşük       :", matematik.min())
print("En yüksek      :", matematik.max())
print("Standart sapma :", round(matematik.std(), 2))
print("Medyan         :", np.median(matematik))

=== Matematik Notu İstatistikleri ===
Ortalama       : 67.75
En düşük       : 35
En yüksek      : 100
Standart sapma : 18.72
Medyan         : 68.0


In [12]:
# axis parametresi: 2D matriste satır mı sütun mu yönünde işlem yapılacağını belirler
# axis=0 -> sütun bazında (her dersin ortalaması)
# axis=1 -> satır bazında (her öğrencinin ortalaması)

ders_ortalamalari = notlar.mean(axis=0)
print("Ders ortalamaları [Matematik, Fen, İngilizce]:", np.round(ders_ortalamalari, 2))

ogrenci_ortalamalari = notlar.mean(axis=1)
print("İlk 5 öğrencinin kişisel not ortalaması:", np.round(ogrenci_ortalamalari[:5], 2))

Ders ortalamaları [Matematik, Fen, İngilizce]: [67.75 66.9  67.78]
İlk 5 öğrencinin kişisel not ortalaması: [50.33 53.   71.67 64.33 41.  ]


In [13]:
# argmax: en büyük değerin İNDEKSİNİ verir -> en başarılı öğrenciyi bulalım
en_iyi_indeks = final_yuzde.argmax()
print("En yüksek final yüzdesine sahip öğrencinin indeksi:", en_iyi_indeks)
print("Öğrenci ID'si  :", satirlar[en_iyi_indeks][0])
print("Final yüzdesi  : %", final_yuzde[en_iyi_indeks])
print("Notları [M,F,İ]:", notlar[en_iyi_indeks])
print("Günlük çalışma :", calisma_saat[en_iyi_indeks], "saat")

En yüksek final yüzdesine sahip öğrencinin indeksi: 21
Öğrenci ID'si  : S0022
Final yüzdesi  : % 98.33
Notları [M,F,İ]: [100  96  99]
Günlük çalışma : 2.8 saat


## 6. Broadcasting ile Hesaplamalar

Derste skaler ile array arasındaki işlemlerin tüm elemanlara otomatik uygulandığını görmüştük.

In [14]:
# Diyelim ki tüm matematik notlarına 5 puan kaynak eklendi (broadcasting)
mat_kaynakli = matematik + 5
# Ama not 100'ü geçemez -> np.clip ile sınırlıyoruz
mat_kaynakli = np.clip(mat_kaynakli, 0, 100)

print("Önce  (ilk 10):", matematik[:10])
print("Sonra (ilk 10):", mat_kaynakli[:10])
print("Yeni ortalama :", round(mat_kaynakli.mean(), 2), "(eski:", round(matematik.mean(), 2), ")")

Önce  (ilk 10): [40 80 83 68 41 88 75 93 55 72]
Sonra (ilk 10): [45 85 88 73 46 93 80 98 60 77]
Yeni ortalama : 72.54 (eski: 67.75 )


In [15]:
# Ağırlıklı ortalama hesabı: Matematik %50, Fen %30, İngilizce %20 ağırlıklı olsun
agirliklar = np.array([0.5, 0.3, 0.2])

# (5000,3) matris * (3,) vektör -> broadcasting ile her satır ağırlıklarla çarpılır
agirlikli_ortalama = (notlar * agirliklar).sum(axis=1)

print("İlk 5 öğrencinin ağırlıklı ortalaması:", np.round(agirlikli_ortalama[:5], 2))
print("Genel ağırlıklı ortalama:", round(agirlikli_ortalama.mean(), 2))

İlk 5 öğrencinin ağırlıklı ortalaması: [46.1 60.2 75.2 63.8 41.5]
Genel ağırlıklı ortalama: 67.5


## 7. Reshape ve Rastgele Örneklem

Son olarak derste gördüğümüz `reshape` ve `np.random` modülünü uyguluyoruz.

In [16]:
# İlk 12 öğrencinin matematik notunu 3x4'lük bir matrise dönüştürelim
ilk12 = matematik[:12]
print("Orijinal (1D):", ilk12)
print("3x4 matris:\n", ilk12.reshape(3, 4))
print("4x3 matris:\n", ilk12.reshape(4, 3))

Orijinal (1D): [40 80 83 68 41 88 75 93 55 72 95 54]
3x4 matris:
 [[40 80 83 68]
 [41 88 75 93]
 [55 72 95 54]]
4x3 matris:
 [[40 80 83]
 [68 41 88]
 [75 93 55]
 [72 95 54]]


In [17]:
# Rastgele örneklem: 5000 öğrenciden rastgele 10 tanesini seçelim
np.random.seed(42)   # tekrarlanabilirlik için tohum sabitliyoruz (derste öğrendik!)

rastgele_indeksler = np.random.choice(matematik.size, size=10, replace=False)  # tekrarsız 10 indeks
print("Seçilen indeksler:", rastgele_indeksler)
print("Bu öğrencilerin matematik notları:", matematik[rastgele_indeksler])
print("Örneklem ortalaması:", matematik[rastgele_indeksler].mean())
print("Tüm veri ortalaması:", round(matematik.mean(), 2))

Seçilen indeksler: [1501 2586 2653 1055  705  106  589 2468 2413 1600]
Bu öğrencilerin matematik notları: [76 36 43 94 79 93 99 59 73 51]
Örneklem ortalaması: 70.3
Tüm veri ortalaması: 67.75


## Özet

Bu pratikte, NumPy Temelleri dersinde öğrendiklerimizi 5.000 kişilik gerçek bir öğrenci veri setine uyguladık:

- CSV verisini **NumPy array'lerine** dönüştürdük
- `shape`, `dtype` gibi **array özelliklerini** inceledik
- **İndeksleme ve dilimleme** ile belirli öğrencilere/derslere ulaştık
- **Boolean indeksleme** ile filtreler kurduk (80+ alanlar, çok çalışanlar, kalanlar...)
- **Aggregate fonksiyonlarla** (mean, min, max, std, argmax) istatistik çıkardık
- **Broadcasting** ile kaynak puanı ve ağırlıklı ortalama hesapladık
- **Reshape** ve **rastgele örneklem** uyguladık

📌 Öne çıkan bulgular:
- Günde 4+ saat çalışan öğrencilerin başarı ortalaması, 2 saatten az çalışanlardan belirgin şekilde yüksek
- Kalan öğrencilerin devam yüzdesi, geçenlere göre daha düşük